In [27]:
select top 5  *
from ecommerce_raw;

(5 rows affected)

order_id         | order_date       | ship_date    | delivery_date    | customer_id | customer_name     | customer_email                | customer_segment | sales_channel | country        | region | product_id  | product_name              | category | subcategory | quantity | unit_price | unit_cogs | discount_pct | gross_revenue | net_revenue | cogs_total | gross_profit | gross_margin_pct | shipping_cost | return_flag | return_reason | return_date | refund_amount | inventory_on_hand | reorder_point | days_in_stock | supplier_id | supplier_lead_time_days | payment_method | payment_status | fulfillment_status | order_status | warehouse_id | notes         
-----------------+------------------+--------------+------------------+-------------+-------------------+-------------------------------+------------------+---------------+----------------+--------+-------------+---------------------------+----------+-------------+----------+------------+-----------+--------------+---

In [ ]:
SELECT DISTINCT (payment_status)
from ecommerce_raw



In [26]:
 UPDATE ecommerce_raw
    SET gross_margin_pct =  cast(REPLACE(gross_margin_pct,'%','') AS DECIMAL(10,2)),
    discount_pct =  cast(REPLACE(discount_pct,'%','') AS DECIMAL(10,2))
    
    ALTER TABLE ecommerce_raw
alter COLUMN gross_margin_pct decimal(10,2) NULL
ALTER TABLE ecommerce_raw
alter COLUMN discount_pct decimal(10,2) NULL


SELECT DISTINCT top 20  product_id,order_id,
            COALESCE(
            TRY_CAST(TRY_CONVERT(DATETIME,order_date,101) AS DATE),
                TRY_CAST(TRY_CONVERT(DATETIME,order_date,103) AS DATE),
                TRY_CAST(TRY_CONVERT(DATETIME,order_date,105) AS DATE),
                TRY_CAST(TRY_CONVERT(DATETIME,order_date,120) AS DATE)
            )AS order_date,
            COALESCE(
            TRY_CAST(TRY_CONVERT(DATETIME,ship_date,101) AS DATE),
                TRY_CAST(TRY_CONVERT(DATETIME,ship_date,103) AS DATE),
                TRY_CAST(TRY_CONVERT(DATETIME,ship_date,105) AS DATE),
                TRY_CAST(TRY_CONVERT(DATETIME,ship_date,120) AS DATE)
            )AS ship_date,
            COALESCE(
            TRY_CAST(TRY_CONVERT(DATETIME,delivery_date,101) AS DATE),
                TRY_CAST(TRY_CONVERT(DATETIME,delivery_date,103) AS DATE),
                TRY_CAST(TRY_CONVERT(DATETIME,delivery_date,105) AS DATE),
                TRY_CAST(TRY_CONVERT(DATETIME,delivery_date,120) AS DATE)
            )AS delivery_date,
            customer_id,
            sales_channel,
            geography_key
            quantity,
            unit_price,
            unit_cogs,
            CASE 
                WHEN discount_pct > 1 THEN discount_pct / 100
                ELSE discount_pct
                END AS discount_pct,
            gross_revenue,
            net_revenue,
            cogs_total,
            gross_profit,
            CASE 
                WHEN  gross_margin_pct > 1 THEN gross_margin_pct / 100
                ELSE gross_margin_pct
                END AS gross_margin_pct,
            shipping_cost,
            
            CASE 
                WHEN lower(payment_method) IN('visa','mastercard','cc') THEN 'Credit Card'
                    ELSE payment_method
                    END AS payment_method,
            payment_status,
            fulfillment_status,
            order_status,
            notes
            


        FROM ecommerce_raw
        LEFT JOIN DimGeography on ecommerce_raw.region = DimGeography.region

(100000 rows affected)
(20 rows affected)

product_id  | order_id         | order_date | ship_date  | delivery_date | customer_id | sales_channel | quantity | unit_price | unit_cogs | discount_pct | gross_revenue | net_revenue | cogs_total | gross_profit | gross_margin_pct | shipping_cost | payment_method | payment_status | fulfillment_status | order_status | notes          
------------+------------------+------------+------------+---------------+-------------+---------------+----------+------------+-----------+--------------+---------------+-------------+------------+--------------+------------------+---------------+----------------+----------------+--------------------+--------------+----------------
SKU-KIT-028 | ORD-482615-15795 | 2024-12-15 | 2024-12-17 | 2024-12-21    | ff3aa69b    | Shopify       | 4        | 91.94      | 48.51     | 0.000000     | 91.94         | 91.94       | 48.51      | 43.43        | 0.470000         | 18.14         | Credit Card    | Paid           | Fulf

In [ ]:
(SELECT
            CASE 
            WHEN  GMP > 1 THEN GMP / 100
            ELSE GMP
            END AS gross_margin_pct

from (SELECT cast(REPLACE(gross_margin_pct,'%','') AS DECIMAL(10,2)) AS GMP
        from ecommerce_raw) as normalized)




SELECT top 20 discount_pct

        from ((SELECT 
            CASE 
                WHEN disc_pct > 1 THEN disc_pct / 100
                ELSE disc_pct
                END AS discount_pct)

        (SELECT 
    
             cast(REPLACE(discount_pct,'%','') AS DECIMAL(10,2)) as disc_pct
    

                from ecommerce_raw) as normalized)
from ecommerce_raw



In [ ]:
WITH normalized as (SELECT gross_margin_pct, discount_pct,
    cast(REPLACE(gross_margin_pct,'%','') AS DECIMAL(10,2)) AS GMP,
    
    cast(REPLACE(discount_pct,'%','') AS DECIMAL(10,2)) as disc_pct
    

        from ecommerce_raw)

SELECT CASE 
            WHEN  GMP > 1 THEN GMP / 100
            ELSE GMP
            END AS gross_margin_pct,
            CASE 
                WHEN disc_pct > 1 THEN disc_pct / 100
                ELSE disc_pct
                END AS discount_pct
            
from normalized;

    

In [ ]:
select DISTINCT(return_flag)
from ecommerce_raw

In [ ]:
select count(distinct(customer_id))
from ecommerce_raw

In [ ]:
select count(DISTINCT(product_id))
from ecommerce_raw

In [ ]:
WITH normalized_data AS (
    SELECT 
        region,
        CASE 
            WHEN country IN ('US', 'United States', 'USA') THEN 'United States'
            ELSE country
        END AS country_normalized
    FROM  ecommerce_raw
    WHERE region IS NOT NULL
),

 DimGeography AS (
    SELECT TOP 20 
    ROW_NUMBER() OVER(ORDER BY country_normalized) AS geography_key,
    region,
    country_normalized

FROM  normalized_data
GROUP BY 
    region,
    country_normalized)

SELECT *
from DimGeography;

In [ ]:
select TOP 20 region,country,ROW_NUMBER() over(order by region,country) as geography_key
from ecommerce_raw
where region IS NOT NULL
group by region,country

In [ ]:
select top 20 CASE 
        WHEN sales_channel in('shop','Shopify' ) THEN 'Shopify'
        when sales_channel in ('AMZ','Amazon' )     THEN 'Amazon'
        ELSE sales_channel
        END AS sales_channel
from ecommerce_raw